# 🟡 Solution: Implement Focal Loss

$$FL(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$$

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    """
    Focal Loss，用于处理类别不平衡的分类任务。

    核心思想：降低简单样本（高置信度预测）的损失权重，
    让模型专注于困难样本。由 RetinaNet 在目标检测中提出。

    公式: FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    其中 p_t 是模型对正确类别的预测概率。

    参数:
        logits: 原始类别分数, shape = (N, C)
            N = 样本数, C = 类别数
        targets: 整数类别索引, shape = (N,)
        alpha: 加权因子 (标量)，平衡正负样本
        gamma: 聚焦参数，越大对简单样本降权越强

    返回: 标量均值损失
    """
    N, C = logits.shape

    # ---- Step 1: 数值稳定的 Softmax ----
    # 直接 exp(logits) 可能溢出，先减去每行最大值
    # max(dim=-1, keepdim=True).values: (N, 1)
    # shifted: (N, C)，每行最大值变为 0，其余为负数
    shifted = logits - logits.max(dim=-1, keepdim=True).values

    # exp(shifted): (N, C)，所有值 <= 1，不会溢出
    exp_s = torch.exp(shifted)

    # 归一化得到概率: (N, C)
    # 每行除以该行的 exp 之和
    probs = exp_s / exp_s.sum(dim=-1, keepdim=True)

    # ---- Step 2: 提取正确类别的概率 p_t ----
    # torch.arange(N): [0, 1, ..., N-1] 作为行索引
    # targets: 每个样本的正确类别，作为列索引
    # 高级索引: probs[行索引, 列索引] 取出对角线元素
    p_t = probs[torch.arange(N), targets]  # (N,)

    # ---- Step 3: 计算 Focal Loss ----
    # log(p_t + 1e-8): 加 epsilon 防止 log(0) = -inf
    # (1 - p_t)^gamma: 聚焦因子
    #   - 当 p_t 接近 1 (简单样本): (1-p_t)^gamma → 0，损失被大幅降低
    #   - 当 p_t 接近 0 (困难样本): (1-p_t)^gamma → 1，损失保持不变
    log_p_t = torch.log(p_t + 1e-8)
    fl = -alpha * (1 - p_t) ** gamma * log_p_t

    # ---- Step 4: 批次均值 ----
    return fl.mean()

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
logits = torch.randn(8, 4)
targets = torch.randint(0, 4, (8,))

# gamma=0 should match weighted CE (alpha * CE)
fl_gamma0 = focal_loss(logits, targets, alpha=0.25, gamma=0.0)
ce = F.cross_entropy(logits, targets)
print(f"Focal (gamma=0): {fl_gamma0:.4f}  |  alpha * CE: {0.25 * ce:.4f}")

# Higher gamma should reduce loss on easy examples
fl_g2 = focal_loss(logits, targets, alpha=0.25, gamma=2.0)
fl_g5 = focal_loss(logits, targets, alpha=0.25, gamma=5.0)
print(f"Focal gamma=2: {fl_g2:.4f}  |  gamma=5: {fl_g5:.4f}  (higher gamma -> lower loss on easy examples)")

In [ ]:
from torch_judge import check
check("focal_loss")

## 📝 核心思路总结

1. **Focal Loss 的核心**：`(1-p_t)^gamma` 聚焦因子让简单样本（p_t 大）的损失趋近 0，迫使模型关注困难样本。
2. **gamma 的作用**：gamma=0 等价于加权 CE；gamma=2（默认）对 p_t>0.5 的样本大幅降权；gamma 越大聚焦效果越强。
3. **数值稳定 softmax**：先减最大值再 exp，是 PyTorch 内部 softmax 的标准实现，面试高频考点。
4. **高级索引取对角线**：`probs[arange(N), targets]` 是 PyTorch 中取出「每个样本对应类别概率」的标准技巧。